In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

X_train, X_test, y_train, y_test = joblib.load("../data/splits.joblib")
results = []

def evaluate(name, model):
    start = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "AUPRC": average_precision_score(y_test, proba),
        "Seconds": round(time.time() - start),
    })

In [3]:
evaluate("Random Forest", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))

In [4]:
evaluate("XGBoost", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                  n_jobs=-1, random_state=42))

In [5]:
evaluate("LightGBM", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    n_jobs=-1, random_state=42, verbose=-1))

In [6]:
table = pd.DataFrame(results).round(3).sort_values("AUPRC", ascending=False)
print(table)

           Model  Precision  Recall     F1  AUPRC  Seconds
1        XGBoost      0.909   0.816  0.860  0.877        1
0  Random Forest      0.941   0.816  0.874  0.873       25
2       LightGBM      0.907   0.796  0.848  0.861        2
